# Kármán vortex street at Reynolds number 100

**Unverified product example.** This uses the Schäfer–Turek 2D-2 geometry and physical parameters, then advances a MINI/P1 and backward-Euler realization. It demonstrates an alternating wake over a sampled window; it does not claim benchmark validation because the reference discretization and refinement study differ.


In [ ]:
# @title Eqiora を準備
%pip install -q "eqiora[gmsh]==0.1.2"

from eqiora.colab import prepare
prepare()


In [ ]:
from importlib.resources import files

import eqiora
import numpy as np

rho = 1.0
mu = 1.0e-3
diameter = 0.1
inlet_maximum = 1.5
inlet_mean = 1.0
dt = 0.01

graph = eqiora.geometry.GeometryGraph()
channel = graph.rectangle(x_bounds=(0.0, 2.2), y_bounds=(0.0, 0.41))
cylinder = graph.circle(center=(0.2, 0.2), radius=diameter / 2.0)
fluid = graph.subtract(channel, cylinder)
geometry = graph.build(fluid, named_topology={
    "fluid": fluid.region,
    "inlet": channel.boundaries[0],
    "outlet": channel.boundaries[1],
    "walls": channel.boundaries[2:4],
    "cylinder": cylinder.boundaries[0],
})


In [ ]:
mesh = eqiora.meshing.generate(eqiora.meshing.resolve(
    geometry,
    eqiora.meshing.GmshMesher(
        maximum_boundary_error=1.0e-4,
        maximum_target_size=0.02,
        minimum_mean_ratio=1.0e-5,
        maximum_boundary_facets=50,
    ),
))
print(mesh.vertex_count, "vertices", mesh.cell_count, "cells")


In [ ]:
source_root = files(eqiora).joinpath("examples")
parameters = {
    "dynamic_viscosity": mu,
    "zero_pressure": 0.0,
    "inlet_speed": inlet_maximum,
    "channel_height": 0.41,
}
supports = {
    "fluid": geometry.selection("fluid"),
    **{
        name: (geometry.selection(name), geometry.selection("fluid"))
        for name in ("inlet", "outlet", "walls", "cylinder")
    },
}
linear = eqiora.solve.Linear(
    algorithm=eqiora.solve.LinearSolver.SparseLu,
    preconditioner=eqiora.solve.Preconditioner.Identity,
    reduction=eqiora.solve.Reduction.Fast,
    provider=eqiora.solve.SolverProvider.faer(),
    relative_tolerance=1.0e-6,
    absolute_tolerance=1.0e-12,
    maximum_iterations=20_000,
)
steady_model = eqiora.compile(
    path=source_root.joinpath("steady-flow-past-cylinder.eqi"),
    geometry=geometry,
    entry="SteadyFlowPastCylinder",
    bindings={**supports, **parameters},
)
steady_plan = eqiora.resolve(
    steady_model, mesh=mesh, spatial=eqiora.fem.MiniP1(), solve=linear,
)
steady_result = eqiora.run(steady_plan)


In [ ]:
model = eqiora.compile(
    path=source_root.joinpath("transient-flow-past-cylinder.eqi"),
    geometry=geometry,
    entry="TransientFlowPastCylinder",
    bindings={**supports, "density": rho, **parameters},
)
plan = eqiora.resolve(
    model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    temporal=eqiora.time.BackwardEuler(dt),
    solve=eqiora.solve.Newton(linear=linear),
    scaling=eqiora.fluid.IncompressibleScaling(
        length_m=diameter,
        velocity_m_per_s=inlet_mean,
        pressure_pa=rho * inlet_mean**2,
    ),
)
velocity = steady_result.output(steady_plan.capability.velocity)
pressure = steady_result.output(steady_plan.capability.pressure)
state = eqiora.State.initial(plan, time_s=0.0, fields=(
    eqiora.InitialField(
        plan.capability.velocity,
        vertex_values=np.asarray(velocity.values("vertex")).reshape(mesh.vertex_count, 2),
        cell_values=np.asarray(velocity.values("cell-bubble")).reshape(mesh.cell_count, 2),
    ),
    eqiora.InitialField(
        plan.capability.pressure,
        vertex_values=np.asarray(pressure.values("vertex")),
    ),
))


In [ ]:
# Keep only each chunk endpoint while the wake develops.
for chunk in range(7):
    spin_up = eqiora.run(plan, state=state, steps=100, output_steps=(100,))
    state = spin_up.trajectory.state(100)
    print(f"spin-up {state.time_s:.2f} s")


In [ ]:
# Retain 100 frames over the next two physical seconds.
result = eqiora.run(
    plan,
    state=state,
    steps=200,
    output_steps=tuple(range(2, 201, 2)),
    profile=True,
)
states = result.trajectory.states
accepted = states[-1]
vorticity = accepted.curl(plan.capability.velocity)

forces = np.asarray([
    item.boundary_force(geometry.selection("cylinder")).on_selection
    for item in states
])
coefficient_scale = 2.0 / (rho * inlet_mean**2 * diameter)
drag = coefficient_scale * forces[:, 0]
lift = coefficient_scale * forces[:, 1]
times = np.asarray([item.time_s for item in states])
pressure_difference = np.asarray([
    item.sample(plan.capability.pressure, at=(0.15, 0.2)).value
    - item.sample(plan.capability.pressure, at=(0.25, 0.2)).value
    for item in states
])
centered_lift = lift - lift.mean()
rising = np.flatnonzero((centered_lift[:-1] <= 0.0) & (centered_lift[1:] > 0.0))
crossings = [
    times[i] - centered_lift[i] * (times[i + 1] - times[i])
    / (centered_lift[i + 1] - centered_lift[i])
    for i in rising
]
strouhal = diameter / (inlet_mean * np.median(np.diff(crossings))) if len(crossings) >= 3 else None
print("C_D", float(drag.min()), float(drag.max()))
print("C_L", float(lift.min()), float(lift.max()))
print("pressure difference", float(pressure_difference.min()), float(pressure_difference.max()), "Pa")
print("sampled Strouhal number", strouhal)
print(result.profile.summary())


In [ ]:
view = eqiora.View().add(geometry).add(mesh).add(vorticity)
view.show()
